In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

In [3]:
df = pd.DataFrame({
    'age': [25, None, 40, 35, None],
    'income': [30000, 45000, None, 80000, 52000],
    'city': ['Chennai', 'Mumbai', None, 'Chennai', 'Delhi'],
    'size': ['S', 'L', 'M', 'S', 'L']
})

print("===== ORIGINAL DATAFRAME =====")
print(df)

print("\n===== MISSING VALUES =====")
print(df.isnull().sum())

===== ORIGINAL DATAFRAME =====
    age   income     city size
0  25.0  30000.0  Chennai    S
1   NaN  45000.0   Mumbai    L
2  40.0      NaN      NaN    M
3  35.0  80000.0  Chennai    S
4   NaN  52000.0    Delhi    L

===== MISSING VALUES =====
age       2
income    1
city      1
size      0
dtype: int64


In [4]:
df_simple = df.copy()

imputer = SimpleImputer(strategy='median')

df_simple[['age']] = imputer.fit_transform(df_simple[['age']])

print("===== AFTER AGE MEDIAN IMPUTATION =====")
print(df_simple)

===== AFTER AGE MEDIAN IMPUTATION =====
    age   income     city size
0  25.0  30000.0  Chennai    S
1  35.0  45000.0   Mumbai    L
2  40.0      NaN      NaN    M
3  35.0  80000.0  Chennai    S
4  35.0  52000.0    Delhi    L


In [5]:
imputer_city = SimpleImputer(strategy='most_frequent')

df_simple[['city']] = imputer_city.fit_transform(df_simple[['city']])

print("===== AFTER CITY IMPUTATION =====")
print(df_simple)

===== AFTER CITY IMPUTATION =====
    age   income     city size
0  25.0  30000.0  Chennai    S
1  35.0  45000.0   Mumbai    L
2  40.0      NaN  Chennai    M
3  35.0  80000.0  Chennai    S
4  35.0  52000.0    Delhi    L


In [6]:
df_knn = df.copy()

knn_imputer = KNNImputer(n_neighbors=2)

df_knn[['age', 'income']] = knn_imputer.fit_transform(
    df_knn[['age', 'income']]
)

print("===== AFTER KNN IMPUTATION =====")
print(df_knn)

===== AFTER KNN IMPUTATION =====
    age   income     city size
0  25.0  30000.0  Chennai    S
1  30.0  45000.0   Mumbai    L
2  40.0  55000.0      NaN    M
3  35.0  80000.0  Chennai    S
4  30.0  52000.0    Delhi    L


In [7]:
comparison = pd.DataFrame({
    'SimpleImputer_Age': df_simple['age'],
    'KNNImputer_Age': df_knn['age'],
    'SimpleImputer_Income': df['income'],
    'KNNImputer_Income': df_knn['income']
})

print("===== IMPUTATION COMPARISON =====")
print(comparison)

===== IMPUTATION COMPARISON =====
   SimpleImputer_Age  KNNImputer_Age  SimpleImputer_Income  KNNImputer_Income
0               25.0            25.0               30000.0            30000.0
1               35.0            30.0               45000.0            45000.0
2               40.0            40.0                   NaN            55000.0
3               35.0            35.0               80000.0            80000.0
4               35.0            30.0               52000.0            52000.0


In [8]:
onehot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

city_encoded = onehot.fit_transform(df_knn[['city']])

city_columns = onehot.get_feature_names_out(['city'])

city_df = pd.DataFrame(
    city_encoded,
    columns=city_columns,
    index=df_knn.index
)

print("===== CITY AFTER ONEHOT ENCODING =====")
print(city_df)

===== CITY AFTER ONEHOT ENCODING =====
   city_Chennai  city_Delhi  city_Mumbai  city_nan
0           1.0         0.0          0.0       0.0
1           0.0         0.0          1.0       0.0
2           0.0         0.0          0.0       1.0
3           1.0         0.0          0.0       0.0
4           0.0         1.0          0.0       0.0


In [9]:
ordinal = OrdinalEncoder(categories=[['S', 'M', 'L']])

df_knn[['size']] = ordinal.fit_transform(df_knn[['size']])

print("===== SIZE AFTER ORDINAL ENCODING =====")
print(df_knn[['size']])

===== SIZE AFTER ORDINAL ENCODING =====
   size
0   0.0
1   2.0
2   1.0
3   0.0
4   2.0


In [10]:
target = pd.Series([
    'Yes', 'No', 'Yes', 'No', 'Yes'
])

label_encoder = LabelEncoder()

target_encoded = label_encoder.fit_transform(target)

print("===== LABEL ENCODER =====")
print("Original Target:")
print(target.tolist())

print("\nEncoded Target:")
print(target_encoded)

print("\nClass Mapping:")
print(dict(zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
)))

===== LABEL ENCODER =====
Original Target:
['Yes', 'No', 'Yes', 'No', 'Yes']

Encoded Target:
[1 0 1 0 1]

Class Mapping:
{'No': np.int64(0), 'Yes': np.int64(1)}


In [11]:
final_df = pd.concat([
    df_knn[['age', 'income']],
    city_df,
    df_knn[['size']]
], axis=1)

print("===== FINAL ENCODED DATAFRAME =====")
print(final_df)

print("\n===== FINAL DATA TYPES =====")
print(final_df.dtypes)

print("\n===== FINAL MISSING VALUES =====")
print(final_df.isnull().sum())

===== FINAL ENCODED DATAFRAME =====
    age   income  city_Chennai  city_Delhi  city_Mumbai  city_nan  size
0  25.0  30000.0           1.0         0.0          0.0       0.0   0.0
1  30.0  45000.0           0.0         0.0          1.0       0.0   2.0
2  40.0  55000.0           0.0         0.0          0.0       1.0   1.0
3  35.0  80000.0           1.0         0.0          0.0       0.0   0.0
4  30.0  52000.0           0.0         1.0          0.0       0.0   2.0

===== FINAL DATA TYPES =====
age             float64
income          float64
city_Chennai    float64
city_Delhi      float64
city_Mumbai     float64
city_nan        float64
size            float64
dtype: object

===== FINAL MISSING VALUES =====
age             0
income          0
city_Chennai    0
city_Delhi      0
city_Mumbai     0
city_nan        0
size            0
dtype: int64
